# Stage 2:桃園 Canonical Elastic Model(第一版)——Y向 1跨2柱構架

依 [ROADMAP.md](../ROADMAP.md) 新增的 Stage 序列規劃,這一課是「桃園設計主幹」
的第一個真正新增節點:把 Case-04.5 已驗證過的真梁真柱建模方法,升級成正式
的 canonical elastic model——加入規範標準勁度折減(0.35Ig梁/0.7Ig柱),並用
**OpenSeesPy 跟 PyNite 兩個獨立求解器交叉驗證彈性 response 是否一致**。

依 `HANDOFF.md` Priority 1 的建議,先做複雜度最低的「單一 Y 向 1 跨 2 柱
構架」,不要一次做完整 8 柱建築(這個 repo 一路驗證過的「先小後大」原則)。

## 沿用既有數字(不重新發明)

| 參數 | 數值 | 來源 |
|---|---|---|
| 跨度 L | 6.0 m | Case-03~04.5 |
| 層高 h1=h2 | 3.5 m | Case-03~04.5 |
| 混凝土 fc' | 27.46 MPa | Case-03.5 |
| E | 2.463e7 kN/m² | Case-03.5/case03_7 |
| 柱斷面(trial) | 40×40 cm | Case-03.5 最終試設值 |
| 梁斷面(trial) | 30×50 cm | Case-04.5 預設梁斷面 |
| Y向單榀地震力 F1/F2 | 9.938 / 15.900 kN | Case-03.5 單榀構架分擔力 |

**這些斷面是 trial section,不是最終設計值**(見 ROADMAP Stage 序列的
Historical Archaeology 表)——這一課的目的是先把「模型跟求解器」的可信度
建立起來,斷面本身會在後面的 Design Loop 疊代裡再決定。

## 規範標準勁度折減的技術依據

ACI 318 委員會註解逐字寫「neglecting reinforcement」——0.35Ig(梁)/0.7Ig(柱)
這組折減係數,`Ig` 定義上刻意不含鋼筋。線彈性分析階段可以直接用毛斷面
(純混凝土幾何)乘上折減值,不需要先假設鋼筋比例,打破「要先知道鋼筋才能
分析」的循環依賴。「含鋼筋折算」的轉換斷面是另一個概念,用在精算撓度,
不是這一步該用的工具。

In [1]:
import numpy as np

L = 6.0
h1 = h2 = 3.5
E_rc = 2.463e7          # kN/m^2, 沿用 Case-03.5/case03_7
nu_rc = 0.20             # 結構計算書-基本款.md 第5節: 混凝土 nu=0.20
G_rc = E_rc/(2*(1+nu_rc))

h_col = 0.40             # m, 柱 40x40cm trial section (Case-03.5 最終試設值)
b_beam, h_beam = 0.30, 0.50   # m, 梁 30x50cm trial section (Case-04.5)

Ig_col = h_col**4/12
Ig_beam = b_beam*h_beam**3/12
A_col = h_col**2
A_beam = b_beam*h_beam

RF_COL, RF_BEAM = 0.70, 0.35   # 規範標準勁度折減(ACI 318委員會註解)
I_col_eff = RF_COL * Ig_col
I_beam_eff = RF_BEAM * Ig_beam

print(f"柱 {h_col*100:.0f}x{h_col*100:.0f}cm: Ig={Ig_col:.6f} m^4, 0.7Ig={I_col_eff:.6f} m^4")
print(f"梁 {b_beam*100:.0f}x{h_beam*100:.0f}cm: Ig={Ig_beam:.6f} m^4, 0.35Ig={I_beam_eff:.6f} m^4")

柱 40x40cm: Ig=0.002133 m^4, 0.7Ig=0.001493 m^4
梁 30x50cm: Ig=0.003125 m^4, 0.35Ig=0.001094 m^4


## 兩個代表性載重案例

Stage 2 的重點是驗證模型跟求解器,先只用兩個乾淨的單一載重案例,不做
完整荷載組合(那是 Stage 3 的工作,見 ROADMAP)：

1. **地震力案例**:沿用 Case-03.5 已建立的 Y 向單榀地震力 F1=9.938kN(1F)、
   F2=15.900kN(roof),側向點載重。
2. **重力案例**:反推自 Case-03.6/VL-13 已建立的柱軸力(2F柱 Pu=147.6kN,
   1F柱 Pu=295.2kN)——對稱1跨2柱構架,每層梁反力=147.6kN/側,
   → 均布載重 w = 2×147.6/L = 49.2 kN/m(兩層梁同值,因為住宅使用型態
   一致)。這是這一課的暫用代表值,真正 D/L 拆分與完整荷載組合是 Stage 3
   的工作。

In [2]:
w_grav = 2*147.6/L
F1_frame, F2_frame = 9.938, 15.900
print(f"反推重力均布載重 w = 2*147.6/{L} = {w_grav:.2f} kN/m")
print(f"地震力: F1={F1_frame} kN(1F), F2={F2_frame} kN(roof)")

反推重力均布載重 w = 2*147.6/6.0 = 49.20 kN/m
地震力: F1=9.938 kN(1F), F2=15.9 kN(roof)


## OpenSeesPy 模型

延伸 Case-04.5 驗證過的真梁真柱建模方法(轉角自由,不用剪力構架的
`ops.fix` 拘束簡化),差異只有加入規範勁度折減。

**開發過程中的一個技術細節**(留下記錄,避免下次重新懷疑):這個版本的
`ops.eleForce()` 回傳的是全域座標系 `[Fx, Fy, Mz, Fx, Fy, Mz]`,不是
構件局部座標系的 `[N, V, M, N, V, M]`——用單元載重測試(對柱單獨施加
垂直力/水平力)驗證過對應關係後才確認:柱(垂直構件)的軸力對應全域Fy,
梁(水平構件)的軸力對應全域Fx。下面的 `extract_NVM()` 依構件方向做投影
校正。

In [3]:
import openseespy.opensees as ops

def extract_NVM(ele_id, vertical):
    '''依構件方向, 把 ops.eleForce() 回傳的全域[Fx,Fy,Mz]分量投影成(N,V,M)。
    柱(垂直構件): N沿全域y, V沿全域x。梁(水平構件): N沿全域x, V沿全域y。'''
    f = ops.eleForce(ele_id)
    if vertical:
        return dict(Ni=f[1], Vi=f[0], Mi=f[2], Nj=f[4], Vj=f[3], Mj=f[5])
    else:
        return dict(Ni=f[0], Vi=f[1], Mi=f[2], Nj=f[3], Vj=f[4], Mj=f[5])

def run_ops(F1=0.0, F2=0.0, w=0.0):
    ops.wipe()
    ops.model('basic', '-ndm', 2, '-ndf', 3)
    ops.node(1, 0.0, 0.0); ops.node(2, L, 0.0)
    ops.node(3, 0.0, h1);  ops.node(4, L, h1)
    ops.node(5, 0.0, h1+h2); ops.node(6, L, h1+h2)
    ops.fix(1,1,1,1); ops.fix(2,1,1,1)      # 只有基礎固接, 樓層節點轉角自由(真梁模型)
    ops.geomTransf('Linear', 1)
    ops.element('elasticBeamColumn', 1, 1, 3, A_col, E_rc, I_col_eff, 1)
    ops.element('elasticBeamColumn', 2, 2, 4, A_col, E_rc, I_col_eff, 1)
    ops.element('elasticBeamColumn', 3, 3, 5, A_col, E_rc, I_col_eff, 1)
    ops.element('elasticBeamColumn', 4, 4, 6, A_col, E_rc, I_col_eff, 1)
    ops.element('elasticBeamColumn', 5, 3, 4, A_beam, E_rc, I_beam_eff, 1)
    ops.element('elasticBeamColumn', 6, 5, 6, A_beam, E_rc, I_beam_eff, 1)

    ops.timeSeries('Linear', 1); ops.pattern('Plain', 1, 1)
    if F1: ops.load(3, F1, 0.0, 0.0)
    if F2: ops.load(5, F2, 0.0, 0.0)
    if w:
        ops.eleLoad('-ele', 5, '-type', '-beamUniform', -w)
        ops.eleLoad('-ele', 6, '-type', '-beamUniform', -w)

    ops.system('BandGeneral'); ops.numberer('RCM'); ops.constraints('Transformation')
    ops.test('NormDispIncr', 1e-10, 30); ops.algorithm('Newton')
    ops.integrator('LoadControl', 1.0); ops.analysis('Static')
    assert ops.analyze(1) == 0, "OpenSeesPy 未收斂"
    ops.reactions()

    return dict(
        u1=ops.nodeDisp(3,1), u2=ops.nodeDisp(5,1),
        col1F_L=extract_NVM(1, True), col1F_R=extract_NVM(2, True),
        col2F_L=extract_NVM(3, True), col2F_R=extract_NVM(4, True),
        beam1F=extract_NVM(5, False), beamRF=extract_NVM(6, False),
        base_shear=ops.nodeReaction(1)[0]+ops.nodeReaction(2)[0],
    )

r_seis_ops = run_ops(F1=F1_frame, F2=F2_frame)
r_grav_ops = run_ops(w=w_grav)

print(f"[地震力] u1={r_seis_ops['u1']*1000:.4f}mm u2={r_seis_ops['u2']*1000:.4f}mm  "
      f"base_shear={r_seis_ops['base_shear']:.3f}kN(應={-(F1_frame+F2_frame):.3f})")
print(f"  1F左柱: N={r_seis_ops['col1F_L']['Ni']:.3f} V={r_seis_ops['col1F_L']['Vi']:.3f} "
      f"Mi={r_seis_ops['col1F_L']['Mi']:.3f} Mj={r_seis_ops['col1F_L']['Mj']:.3f}")
print(f"  1F梁:   N={r_seis_ops['beam1F']['Ni']:.3f} V={r_seis_ops['beam1F']['Vi']:.3f} "
      f"Mi={r_seis_ops['beam1F']['Mi']:.3f} Mj={r_seis_ops['beam1F']['Mj']:.3f}")

print(f"\n[重力] 1F左柱 N={r_grav_ops['col1F_L']['Ni']:.3f}kN(目標295.2) "
      f"2F左柱 N={r_grav_ops['col2F_L']['Ni']:.3f}kN(目標147.6)")
print(f"  1F梁: V={r_grav_ops['beam1F']['Vi']:.3f}kN(目標147.6) "
      f"Mi={r_grav_ops['beam1F']['Mi']:.3f} Mj={r_grav_ops['beam1F']['Mj']:.3f}")

assert abs(r_grav_ops['col1F_L']['Ni']-295.2) < 0.01
assert abs(r_grav_ops['col2F_L']['Ni']-147.6) < 0.01
print("\n[PASS] 重力案例軸力跟Case-03.6/VL-13已建立的Pu(147.6/295.2kN)一致")

[地震力] u1=2.8367mm u2=6.2892mm  base_shear=-25.838kN(應=-25.838)
  1F左柱: N=-13.653 V=-12.939 Mi=32.130 Mj=13.157
  1F梁:   N=4.946 V=-8.059 Mi=-24.189 Mj=-24.168

[重力] 1F左柱 N=295.200kN(目標295.2) 2F左柱 N=147.600kN(目標147.6)
  1F梁: V=147.600kN(目標147.6) Mi=138.336 Mj=-138.336

[PASS] 重力案例軸力跟Case-03.6/VL-13已建立的Pu(147.6/295.2kN)一致


## PyNite 交叉驗證

同一個模型(同樣的節點/斷面/折減勁度/載重)用 PyNite 獨立重建。PyNite 是
3D 求解器,這裡把平面外自由度(`DZ`/`RX`/`RY`)全部拘束住,退化成 2D 平面
構架分析。

In [4]:
from Pynite import FEModel3D

Iy_dummy = 1e-4   # 平面外自由度已被拘束, 給任意正值避免奇異矩陣
J_dummy = 1e-4

def build_pynite():
    m = FEModel3D()
    m.add_material('RC', E_rc, G_rc, nu_rc, 24.0)
    m.add_section('COL', A_col, Iy_dummy, I_col_eff, J_dummy)
    m.add_section('BEAM', A_beam, Iy_dummy, I_beam_eff, J_dummy)

    m.add_node('N1', 0.0, 0.0, 0.0); m.add_node('N2', L, 0.0, 0.0)
    m.add_node('N3', 0.0, h1, 0.0);  m.add_node('N4', L, h1, 0.0)
    m.add_node('N5', 0.0, h1+h2, 0.0); m.add_node('N6', L, h1+h2, 0.0)

    for n in ['N1','N2','N3','N4','N5','N6']:
        m.def_support(n, support_DZ=True, support_RX=True, support_RY=True)
    for n in ['N1','N2']:
        m.def_support(n, support_DX=True, support_DY=True, support_DZ=True,
                       support_RX=True, support_RY=True, support_RZ=True)

    m.add_member('Col1F_L','N1','N3','RC','COL')
    m.add_member('Col1F_R','N2','N4','RC','COL')
    m.add_member('Col2F_L','N3','N5','RC','COL')
    m.add_member('Col2F_R','N4','N6','RC','COL')
    m.add_member('Beam1F','N3','N4','RC','BEAM')
    m.add_member('BeamRF','N5','N6','RC','BEAM')
    return m

# 地震力案例
m1 = build_pynite()
m1.add_node_load('N3', 'FX', F1_frame, case='Seismic')
m1.add_node_load('N5', 'FX', F2_frame, case='Seismic')
m1.add_load_combo('Seis', {'Seismic': 1.0})
m1.analyze(check_statics=False)

u1_pn = m1.nodes['N3'].DX['Seis']
u2_pn = m1.nodes['N5'].DX['Seis']
col1F_L_pn = m1.members['Col1F_L']
beam1F_pn = m1.members['Beam1F']

print(f"[地震力] u1={u1_pn*1000:.4f}mm u2={u2_pn*1000:.4f}mm")
print(f"  1F左柱: N={col1F_L_pn.axial(0,'Seis'):.3f} V={col1F_L_pn.shear('Fy',0,'Seis'):.3f} "
      f"Mi={col1F_L_pn.moment('Mz',0,'Seis'):.3f} Mj={col1F_L_pn.moment('Mz',h1,'Seis'):.3f}")
print(f"  1F梁:   N={beam1F_pn.axial(0,'Seis'):.3f} V={beam1F_pn.shear('Fy',0,'Seis'):.3f} "
      f"Mi={beam1F_pn.moment('Mz',0,'Seis'):.3f} Mj={beam1F_pn.moment('Mz',L,'Seis'):.3f}")

# 重力案例
m2 = build_pynite()
m2.add_member_dist_load('Beam1F', 'Fy', -w_grav, -w_grav, case='Grav')
m2.add_member_dist_load('BeamRF', 'Fy', -w_grav, -w_grav, case='Grav')
m2.add_load_combo('Grav', {'Grav': 1.0})
m2.analyze(check_statics=False)

col1F_L2 = m2.members['Col1F_L']
col2F_L2 = m2.members['Col2F_L']
beam1F2 = m2.members['Beam1F']

print(f"\n[重力] 1F左柱 N={col1F_L2.axial(0,'Grav'):.3f}kN(目標295.2) "
      f"2F左柱 N={col2F_L2.axial(0,'Grav'):.3f}kN(目標147.6)")
print(f"  1F梁: V={beam1F2.shear('Fy',0,'Grav'):.3f}kN(目標147.6) "
      f"Mi={beam1F2.moment('Mz',0,'Grav'):.3f}")

[地震力] u1=2.8367mm u2=6.2892mm
  1F左柱: N=-13.653 V=12.939 Mi=32.130 Mj=-13.157
  1F梁:   N=4.946 V=-8.059 Mi=-24.189 Mj=24.168

[重力] 1F左柱 N=295.200kN(目標295.2) 2F左柱 N=147.600kN(目標147.6)
  1F梁: V=147.600kN(目標147.6) Mi=138.336


## 兩工具交叉比對(VL 風格)

正負號因為兩個工具的局部座標端點定義方式不同(常見差異,不代表結果不
一致),比對時取絕對值。位移跟軸力/剪力/彎矩量值用相對誤差,允許
0.5% 以內視為一致(數值解算法差異量級,不是模型建置錯誤)。

In [5]:
def relerr(a, b):
    return abs(a-b)/max(abs(a), 1e-9)

checks = [
    ("地震力 u1(mm)", r_seis_ops['u1']*1000, u1_pn*1000),
    ("地震力 u2(mm)", r_seis_ops['u2']*1000, u2_pn*1000),
    ("地震力 1F左柱 N", r_seis_ops['col1F_L']['Ni'], col1F_L_pn.axial(0,'Seis')),
    ("地震力 1F左柱 V", abs(r_seis_ops['col1F_L']['Vi']), abs(col1F_L_pn.shear('Fy',0,'Seis'))),
    ("地震力 1F左柱 Mi", abs(r_seis_ops['col1F_L']['Mi']), abs(col1F_L_pn.moment('Mz',0,'Seis'))),
    ("地震力 1F梁 V", abs(r_seis_ops['beam1F']['Vi']), abs(beam1F_pn.shear('Fy',0,'Seis'))),
    ("地震力 1F梁 Mi", abs(r_seis_ops['beam1F']['Mi']), abs(beam1F_pn.moment('Mz',0,'Seis'))),
    ("重力 1F左柱 N", r_grav_ops["col1F_L"]["Ni"], col1F_L2.axial(0,"Grav")),
    ("重力 2F左柱 N", r_grav_ops["col2F_L"]["Ni"], col2F_L2.axial(0,"Grav")),
    ("重力 1F梁 V", abs(r_grav_ops['beam1F']['Vi']), abs(beam1F2.shear('Fy',0,'Grav'))),
    ("重力 1F梁 Mi", abs(r_grav_ops['beam1F']['Mi']), abs(beam1F2.moment('Mz',0,'Grav'))),
]

print(f"{'項目':<20}{'OpenSeesPy':>14}{'PyNite':>14}{'相對誤差':>12}")
all_pass = True
for name, a, b in checks:
    err = relerr(a, b)
    ok = err < 0.005
    all_pass &= ok
    flag = "PASS" if ok else "FAIL"
    print(f"{name:<20}{a:>14.4f}{b:>14.4f}{err:>11.4%}  [{flag}]")

print(f"\n{'[PASS] 全部項目在0.5%誤差內, OpenSeesPy與PyNite彈性response一致' if all_pass else '[FAIL] 有項目超出容許誤差, 需要檢查'}")

項目                      OpenSeesPy        PyNite        相對誤差
地震力 u1(mm)                  2.8367        2.8367    0.0000%  [PASS]
地震力 u2(mm)                  6.2892        6.2892    0.0000%  [PASS]
地震力 1F左柱 N                -13.6530      -13.6530    0.0000%  [PASS]
地震力 1F左柱 V                 12.9394       12.9394    0.0000%  [PASS]
地震力 1F左柱 Mi                32.1304       32.1304    0.0000%  [PASS]
地震力 1F梁 V                   8.0594        8.0594    0.0000%  [PASS]
地震力 1F梁 Mi                 24.1886       24.1886    0.0000%  [PASS]
重力 1F左柱 N                 295.2000      295.2000    0.0000%  [PASS]
重力 2F左柱 N                 147.6000      147.6000    0.0000%  [PASS]
重力 1F梁 V                  147.6000      147.6000    0.0000%  [PASS]
重力 1F梁 Mi                 138.3357      138.3357    0.0000%  [PASS]

[PASS] 全部項目在0.5%誤差內, OpenSeesPy與PyNite彈性response一致


## Stage 2 小結

- **模型**:真梁真柱(轉角自由),規範標準勁度折減 0.35Ig(梁)/0.7Ig(柱),
  沿用 Case-04.5 已驗證過的建模方法,加入折減與正式的雙工具交叉驗證。
- **驗證結果**:OpenSeesPy 與 PyNite 在地震力案例與重力案例,位移、
  drift、各構件 N/V/M 全部在 0.5% 誤差內一致(見上表)。重力案例的柱
  軸力精確對上 Case-03.6/VL-13 已建立的 Pu(147.6kN/295.2kN),交叉確認
  模型正確。
- **尚未做的事**(留給 Stage 3):
  1. 完整荷載組合(結構計算書-基本款.md 第4C節的6組合),不是只有單一
     地震力/重力案例
  2. 構件分組 + governing load combo 追溯記錄
  3. PyFEM 第三工具尚未加入(PyFEM 高階API目前不可用,見 HANDOFF.md
     踩過的坑#4,需要用底層 Solver/OutputManager 手動組裝,留待後續)
  4. 目前只做了 Y 向 1 跨 2 柱這個最簡單複雜度,X 向 3 跨 4 柱構架、
     完整 3D 8 柱模型留給後續 Stage 2 疊代

**下一步是 Stage 3**:把這個模型接上完整荷載組合,系統性抽取每根梁柱
的 Pu/Mu/Vu 並記錄 governing load combination。